In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col

bronze_df = spark.table("bronze_nba_shot_logs")
silver_df = spark.table("silver_nba_shot_logs_cleaned")
gold_df = spark.table("gold_nba_shot_quality_features")

checks = []         # all types of ways to clean data

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
gold_count = gold_df.count()

checks.append(Row(
    check_name="bronze_to_silver_row_count",
    table_name="silver_nba_shot_logs_cleaned",
    check_type="reconciliation",
    failed_records=int(bronze_count - silver_count),
    status="PASS" if silver_count > 0 else "FAIL",
    severity="HIGH"
))

checks.append(Row(
    check_name="silver_to_gold_row_count",
    table_name="gold_nba_shot_quality_features",
    check_type="reconciliation",
    failed_records=int(silver_count - gold_count),
    status="PASS" if silver_count == gold_count else "WARN",
    severity="MEDIUM"
))

In [0]:
# null check

required_cols = [
    "shot_dist",
    "close_def_dist",
    "shot_clock_adjusted",
    "game_clock_seconds_remaining",
    "total_seconds_left_in_game",
    "is_shot_clock_missing",
    "dribbles",
    "touch_time",
    "period",
    "pts_type",
    "label"
]

for c in required_cols:
    null_count = gold_df.filter(col(c).isNull()).count()
    checks.append(Row(
        check_name=f"null_check_{c}",
        table_name="gold_nba_shot_quality_features",
        check_type="null_check",
        failed_records=int(null_count),
        status="PASS" if null_count == 0 else "FAIL",
        severity="HIGH"
    ))

In [0]:
# out of bounds checks

range_checks = [
    ("shot_dist_negative", gold_df.filter(col("shot_dist") < 0).count()),
    ("close_def_dist_negative", gold_df.filter(col("close_def_dist") < 0).count()),
    ("shot_clock_adjusted_out_of_range", gold_df.filter((col("shot_clock_adjusted") < 0) | (col("shot_clock_adjusted") > 24)).count()),
    ("game_clock_seconds_out_of_range", gold_df.filter((col("game_clock_seconds_remaining") < 0) | (col("game_clock_seconds_remaining") > 720)).count()),
    ("total_seconds_left_out_of_range", gold_df.filter((col("total_seconds_left_in_game") < 0) | (col("total_seconds_left_in_game") > 2880)).count()),
    ("dribbles_negative", gold_df.filter(col("dribbles") < 0).count()),
    ("touch_time_negative", gold_df.filter(col("touch_time") < 0).count()),
    ("period_out_of_range", gold_df.filter(~col("period").between(1, 4)).count()),
    ("invalid_label", gold_df.filter(~col("label").isin(0, 1)).count())
]

for check_name, fail_count in range_checks:
    checks.append(Row(
        check_name=check_name,
        table_name="gold_nba_shot_quality_features",
        check_type="range_check",
        failed_records=int(fail_count),
        status="PASS" if fail_count == 0 else "FAIL",
        severity="HIGH"
    ))

In [0]:
binary_feature_cols = [
    "is_shot_clock_missing",

    "is_close_shot",
    "is_mid_range_shot",
    "is_three_point_shot",
    "is_deep_shot",

    "is_very_tight_defense",
    "is_tight_defense",
    "is_open_defense",
    "is_wide_open_defense",

    "is_last_5_min_game",
    "is_last_2_min_period",
    "is_end_of_period",

    "is_very_late_clock",
    "is_late_clock",

    "is_home_game",
    "is_catch_and_shoot"
]

for c in binary_feature_cols:
    invalid_count = gold_df.filter(~col(c).isin(0, 1)).count()
    checks.append(Row(
        check_name=f"binary_feature_check_{c}",
        table_name="gold_nba_shot_quality_features",
        check_type="binary_feature_check",
        failed_records=int(invalid_count),
        status="PASS" if invalid_count == 0 else "FAIL",
        severity="MEDIUM"
    ))

In [0]:
# defender distance checks

defender_bucket_error_count = gold_df.filter(
    (
        col("is_very_tight_defense") +
        col("is_tight_defense") +
        col("is_open_defense") +
        col("is_wide_open_defense")
    ) != 1
).count()

checks.append(Row(
    check_name="defender_distance_bucket_exclusivity",
    table_name="gold_nba_shot_quality_features",
    check_type="feature_logic_check",
    failed_records=int(defender_bucket_error_count),
    status="PASS" if defender_bucket_error_count == 0 else "FAIL",
    severity="HIGH"
))

In [0]:
# shot clock timing checks

shot_clock_bucket_error_count = gold_df.filter(
    (col("is_very_late_clock") + col("is_late_clock")) > 1
).count()

checks.append(Row(
    check_name="shot_clock_bucket_exclusivity",
    table_name="gold_nba_shot_quality_features",
    check_type="feature_logic_check",
    failed_records=int(shot_clock_bucket_error_count),
    status="PASS" if shot_clock_bucket_error_count == 0 else "FAIL",
    severity="MEDIUM"
))

In [0]:
# shot clock disabled checks

shot_clock_missing_rows = gold_df.filter(col("is_shot_clock_missing") == 1).count()

checks.append(Row(
    check_name="shot_clock_missingness_preserved",
    table_name="gold_nba_shot_quality_features",
    check_type="missingness_handling",
    failed_records=0,
    status="PASS",
    severity="LOW"
))

print(f"Rows with originally missing shot clock: {shot_clock_missing_rows}")

In [0]:
# shot distance checks

shot_distance_bucket_error_count = gold_df.filter(
    (
        col("is_close_shot") +
        col("is_mid_range_shot") +
        col("is_three_point_shot") +
        col("is_deep_shot")
    ) != 1
).count()

checks.append(Row(
    check_name="shot_distance_bucket_exclusivity",
    table_name="gold_nba_shot_quality_features",
    check_type="feature_logic_check",
    failed_records=int(shot_distance_bucket_error_count),
    status="PASS" if shot_distance_bucket_error_count == 0 else "FAIL",
    severity="HIGH"
))

In [0]:
make_rate = gold_df.selectExpr("avg(label) as make_rate").collect()[0]["make_rate"]

checks.append(Row(
    check_name="target_class_balance",
    table_name="gold_nba_shot_quality_features",
    check_type="ml_data_quality",
    failed_records=0,
    status="PASS" if make_rate is not None and 0.25 <= make_rate <= 0.65 else "WARN",
    severity="MEDIUM"
))

In [0]:
scorecard_df = spark.createDataFrame(checks)

spark.sql("DROP TABLE IF EXISTS gold_nba_shot_quality_scorecard")
scorecard_df.write.mode("overwrite").saveAsTable("gold_nba_shot_quality_scorecard")

display(scorecard_df)

In [0]:
spark.sql("""
SELECT 
  status,
  severity,
  COUNT(*) AS check_count,
  SUM(failed_records) AS total_failed_records
FROM gold_nba_shot_quality_scorecard
GROUP BY status, severity
ORDER BY severity, status
""").show()